# AI Accelerator Final Project - Colab Runner

이 노트북은 Google Colab에서 MNIST 1/3/5 + 커스텀 6/8 학습과 PGM 평가를 실행합니다.

권장 런타임: `Runtime > Change runtime type > T4 GPU`

## 1. 프로젝트 가져오기

둘 중 하나만 사용하세요.

- GitHub에 올린 경우: `REPO_URL`에 저장소 URL을 넣고 다음 셀을 실행
- Drive에 폴더를 올린 경우: `PROJECT_DIR`을 Drive 안 프로젝트 경로로 바꾸고 `REPO_URL = ""` 유지

In [ ]:
from pathlib import Path
import os

REPO_URL = "https://github.com/eoog333/AI-Accelerator-Final-Project.git"
PROJECT_DIR = Path("/content/AI_accelerator_final_project")

if REPO_URL:
    if not PROJECT_DIR.exists():
        !git clone {REPO_URL} {PROJECT_DIR}
    else:
        %cd {PROJECT_DIR}
        !git fetch origin main
        !git reset --hard origin/main
else:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/AI_accelerator_final_project")

assert PROJECT_DIR.exists(), f"프로젝트 폴더를 찾을 수 없습니다: {PROJECT_DIR}"
os.chdir(PROJECT_DIR)
print("PROJECT_DIR =", PROJECT_DIR)
!find . -maxdepth 2 -type f | sort | sed -n '1,80p'

## 2. 의존성 설치

Colab에는 PyTorch가 기본 설치되어 있으므로 `requirements.txt`를 기준으로 필요한 패키지를 맞춥니다.

In [ ]:
!python -m pip install -q -r requirements.txt

import torch
print("torch", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

## 3. 실행 경로 설정

`CUSTOM_ROOT`는 JPG/PGM 모두 읽을 수 있습니다. 현재 프로젝트의 기본 커스텀 데이터는 `data/custom_digits`입니다.

In [ ]:
CUSTOM_ROOT = "data/custom_digits"
PGM_ROOT = "data/custom_digits_pgm"  # YOLO 결과를 평가하려면 "data/pgm"로 변경
MODEL_OUT = "models/mnist_13568_colab.pt"
METRICS_OUT = "results/mnist_metrics_colab.json"
PGM_METRICS_OUT = "results/pgm_eval_colab.json"

!mkdir -p models results data/mnist
!find {CUSTOM_ROOT} -maxdepth 2 -type f | wc -l
!find {PGM_ROOT} -maxdepth 2 -type f -name '*.pgm' | wc -l

## 4. 빠른 동작 확인

처음 실행할 때 MNIST 파일을 내려받습니다. 아래 셀은 짧은 학습으로 코드와 데이터 경로만 확인합니다.

In [ ]:
!python src/mnist/train_mnist_13568.py \
  --download \
  --data-dir data/mnist \
  --custom-root {CUSTOM_ROOT} \
  --epochs 1 \
  --limit-train 512 \
  --batch-size 128 \
  --model-out models/mnist_13568_smoke.pt \
  --metrics-out results/mnist_metrics_smoke.json

## 5. 전체 학습

제출/실험용 결과는 이 셀을 사용하세요. 필요하면 `EPOCHS`를 조정합니다.

In [ ]:
EPOCHS = 8

!python src/mnist/train_mnist_13568.py \
  --download \
  --data-dir data/mnist \
  --custom-root {CUSTOM_ROOT} \
  --epochs {EPOCHS} \
  --batch-size 128 \
  --model-out {MODEL_OUT} \
  --metrics-out {METRICS_OUT}

## 6. PGM 평가

`PGM_ROOT`가 `data/custom_digits_pgm/6/*.pgm`, `data/custom_digits_pgm/8/*.pgm`처럼 라벨 폴더를 포함하면 digit별 정확도가 출력됩니다.

In [ ]:
!python src/mnist/eval_pgm.py \
  --weights {MODEL_OUT} \
  --pgm-root {PGM_ROOT} \
  --metrics-out {PGM_METRICS_OUT}

## 7. 결과 확인

In [ ]:
import json
from pathlib import Path

for path in [METRICS_OUT, PGM_METRICS_OUT]:
    path = Path(path)
    print("\n==", path, "==")
    if not path.exists():
        print("not found")
        continue
    data = json.loads(path.read_text())
    keys = ["device", "epochs", "total_time_sec", "accuracy", "samples", "labeled_samples", "avg_latency_ms"]
    for key in keys:
        if key in data:
            print(f"{key}: {data[key]}")
    if "mnist_test" in data:
        print("mnist_test accuracy:", data["mnist_test"]["accuracy"])
    if "custom_eval" in data and data["custom_eval"]:
        print("custom_eval accuracy:", data["custom_eval"]["accuracy"])
    if "per_digit" in data:
        print("per_digit:", data["per_digit"])